# 2. Embeddings

An embedding is a fixed-size vector that summarises a DNA sequence; it is the usual input for
downstream models (classifiers, clustering, search). This notebook covers the three forms of
`embed_sequence`, which layer to read, and ends with a small classifier built on embeddings.

> Run the series in order the first time: notebook 01 explains the setup. Every notebook
> starts by connecting to the services through `tutorial_utils.py` next to it; a service
> that isn't running is skipped.

In [1]:
from tutorial_utils import (
    HBB,
    HBB_SICKLE,
    RANDOM_SEQ,
    connect,
    cosine,
    default_checkpoints,
    fetch_hg38,
    show_table,
)

clients = connect()
defaults = default_checkpoints(clients)
print(defaults)

server http://localhost | token set: True | up: ntv3, ntv2, dnabert2, hyenadna, evo2, generator, genalm, grover
{'ntv3': '100m-pre', 'ntv2': '500m', 'dnabert2': '117m', 'hyenadna': 'medium-160k', 'evo2': '7b', 'generator': 'v2-eukaryote-1.2b', 'genalm': 'bert-large-t2t', 'grover': 'grover'}


## Three forms

| Call | Result |
|---|---|
| `embed_sequence("ACGT...")` | `embedding`: one vector (mean over tokens, `pooling="mean"`) |
| `embed_sequence(["ACGT...", "GGCC..."])` | `embeddings`: one vector per sequence, in one call |
| `embed_sequence("ACGT...", pooling="per_token")` | `embedding`: a matrix, one row per token |

A batch is faster than one call per sequence, and sequences in a batch may differ in length:
the server pads them (or runs them one by one where padding would change the result) and
averages only real tokens. Each sequence must fit the model's context limit.

In [2]:
c, ck = clients["ntv2"], defaults["ntv2"]
single = c.embed_sequence(HBB, checkpoint=ck)["embedding"]
batch = c.embed_sequence([HBB, HBB_SICKLE, HBB[:60]], checkpoint=ck)["embeddings"]
per_token = c.embed_sequence(HBB, checkpoint=ck, pooling="per_token")["embedding"]

print("single:   ", len(single), "values")
print(
    "batch:    ",
    len(batch),
    "x",
    len(batch[0]),
    "| batch[0] vs single: cosine",
    round(cosine(batch[0], single), 6),
)
print("per_token:", len(per_token), "rows x", len(per_token[0]))

single:    1024 values
batch:     3 x 1024 | batch[0] vs single: cosine 1.0
per_token: 16 rows x 1024


### What is a row in `per_token`?

That depends on the tokenizer: one row per base for NTv3, HyenaDNA and Evo 2; a leading
`<cls>` plus one row per 6-mer for NTv2; one per 6-mer (plus `<s>` markers) for GENERator;
one per BPE token plus `[CLS]`/`[SEP]` for DNABERT-2, GENA-LM and GROVER. The number of rows
for the same 90 bp shows it:

In [3]:
rows = []
for name, c in clients.items():
    out = c.embed_sequence(HBB, checkpoint=defaults[name], pooling="per_token")["embedding"]
    rows.append(
        {
            "service": name,
            "checkpoint": defaults[name],
            "rows for 90 bp": len(out),
            "dim": len(out[0]),
        }
    )
show_table(rows)

service    checkpoint         rows for 90 bp  dim 
---------  -----------------  --------------  ----
ntv3       100m-pre           90              768 
ntv2       500m               16              1024
dnabert2   117m               22              768 
hyenadna   medium-160k        90              256 
evo2       7b                 90              4096
generator  v2-eukaryote-1.2b  17              2048
genalm     bert-large-t2t     17              1024
grover     grover             25              768 


## Which layer?

By default every service returns the **last layer**, except Evo 2, whose default is the
intermediate layer its authors recommend (`blocks.28.mlp.l3`). Intermediate layers are often
better features for downstream tasks. `get_embedding_layers` lists the valid values for
`layer_name`, with a suggestion (`which="recommended"`):

In [4]:
for name in ["ntv2", "evo2"]:
    if name in clients:
        layers = clients[name].get_embedding_layers(defaults[name], which="recommended")
        print(name, {k: v for k, v in layers.items() if k != "checkpoint"})

ntv2 {'layers': [28, 29], 'num_hidden_state_layers': 30, 'info': "Heuristic only: the final layer and the one before it. InstaDeep's NTv2 model cards use the final layer; use which='all' to explore others."}
evo2 {'layers': ['blocks.28.mlp.l3', 'blocks.26.mlp.l3'], 'bases_per_position': {'blocks.28.mlp.l3': 1, 'blocks.26.mlp.l3': 1}, 'num_hidden_state_layers': 33, 'info': "Evo 2's paper and Arc's examples use intermediate-layer embeddings, not the last layer: for evo2_7b Arc's README embeds with 'blocks.28.mlp.l3' and the paper's sparse autoencoders read layer 26. 'blocks.N.mlp.l3' is block N's MLP output (before the residual add), named as in Arc's evo2 package. For checkpoints without a published choice this is a heuristic at the same relative depth."}


In [5]:
# NTv2 500M has 29 layers: the last one vs layer 20, the layer used in the gene-fusion benchmark paper
c = clients["ntv2"]
last = c.embed_sequence(HBB, checkpoint="500m")
l20 = c.embed_sequence(HBB, checkpoint="500m", layer_name="20")
print(
    "layers:",
    last["layer_name"],
    "and",
    l20["layer_name"],
    "| cosine between them:",
    round(cosine(last["embedding"], l20["embedding"]), 3),
)

layers: 29 and 20 | cosine between them: 0.622


## Mean or a single position?

The mean summarises the whole sequence. When one position matters (a breakpoint, a variant),
the row at that position in `per_token` keeps a local signal that averaging dilutes; that is
the "middle embedding" used in the gene-fusion benchmark paper. With one token per base the
row index is the base position; with 6-mers or BPE, it is the token covering that base.

In [6]:
if "evo2" in clients:
    rows = clients["evo2"].embed_sequence(HBB, pooling="per_token")["embedding"]
    middle = rows[len(HBB) // 2]
    mean = clients["evo2"].embed_sequence(HBB)["embedding"]
    print(f"Evo 2: {len(rows)} rows (one per base); middle row = base {len(HBB) // 2}")
    print("cosine(middle row, mean):", round(cosine(middle, mean), 3))

Evo 2: 90 rows (one per base); middle row = base 45
cosine(middle row, mean): 0.814


## Comparing sequences

`compare_sequences` returns the cosine similarity of two mean embeddings. With these models
almost everything scores high (embeddings sit in a narrow cone), so read it *relatively*: a
1-base mutation should be closer to the original than a random sequence. For search or
clustering over a dataset, centre and standardise the embeddings first.

In [7]:
rows = []
for name, c in clients.items():
    ck = defaults[name]
    rows.append(
        {
            "service": name,
            "cos(HBB, 1-SNP)": c.compare_sequences(HBB, HBB_SICKLE, checkpoint=ck)[
                "cosine_similarity"
            ],
            "cos(HBB, random)": c.compare_sequences(HBB, RANDOM_SEQ, checkpoint=ck)[
                "cosine_similarity"
            ],
        }
    )
show_table(rows)

service    cos(HBB, 1-SNP)  cos(HBB, random)
---------  ---------------  ----------------
ntv3       0.9997           0.2689          
ntv2       0.9983           0.2282          
dnabert2   0.9851           0.2940          
hyenadna   0.9994           0.8481          
evo2       0.9800           0.5748          
generator  0.9996           0.9925          
genalm     0.9936           0.3974          
grover     0.9883           0.2499          


## From embeddings to a classifier

The typical use: embed labelled sequences once, then train a simple model on the vectors. As
a small, self-contained task we ask: **is this real human DNA, or the same bases shuffled?**
Shuffling keeps the base composition but destroys the order, so a model only separates the
two if its embeddings capture sequence structure.

We cut 300 bp windows from the HBB locus (hg38, via the UCSC API, so this needs internet),
make a shuffled copy of each, and classify with a nearest-centroid rule on standardised
embeddings, trained on half of the windows and tested on the other half.

In [8]:
import random

region = fetch_hg38("chr11", 5_200_000, 5_224_000)
windows = [region[i : i + 300] for i in range(0, len(region) - 300 + 1, 600)]
windows = [w for w in windows if "N" not in w][:40]
rng = random.Random(0)
shuffled = ["".join(rng.sample(w, len(w))) for w in windows]
sequences = windows + shuffled
labels = [1] * len(windows) + [0] * len(shuffled)
test = [i % 2 == 1 for i in range(len(sequences))]  # odd windows (and their shuffles) are held out
print(len(sequences), "sequences;", sum(test), "for testing")


def nearest_centroid_accuracy(vectors):
    train = [v for v, t in zip(vectors, test) if not t]
    dims = range(len(vectors[0]))
    mean = [sum(v[d] for v in train) / len(train) for d in dims]
    std = [max((sum((v[d] - mean[d]) ** 2 for v in train) / len(train)) ** 0.5, 1e-6) for d in dims]
    z = [[(v[d] - mean[d]) / std[d] for d in dims] for v in vectors]
    centroids = {}
    for label in (0, 1):
        members = [zv for zv, y, t in zip(z, labels, test) if y == label and not t]
        centroids[label] = [sum(m[d] for m in members) / len(members) for d in dims]
    hits = [
        max((0, 1), key=lambda lab: cosine(zv, centroids[lab])) == y
        for zv, y, t in zip(z, labels, test)
        if t
    ]
    return sum(hits) / len(hits)


rows = []
for name, c in clients.items():
    vectors = c.embed_sequence(sequences, checkpoint=defaults[name])["embeddings"]
    rows.append(
        {
            "service": name,
            "checkpoint": defaults[name],
            "test accuracy": nearest_centroid_accuracy(vectors),
        }
    )
show_table(rows)

80 sequences; 40 for testing


service    checkpoint         test accuracy
---------  -----------------  -------------
ntv3       100m-pre           0.7250       
ntv2       500m               0.8000       
dnabert2   117m               0.8500       
hyenadna   medium-160k        0.7500       
evo2       7b                 0.7500       
generator  v2-eukaryote-1.2b  0.7750       
genalm     bert-large-t2t     0.8750       
grover     grover             0.9750       


A real study would use a proper dataset, cross-validation and a stronger classifier (logistic
regression, SVM, a small neural network), but the workflow is the same: embeddings once,
then any classifier. For long sequences, check each model's context limit in
`list_available_checkpoints`.

**Next:** [3. Variant effects](03_variant_effects.ipynb)